In [ ]:
from imblearn.over_sampling import SMOTE
import cv2
import os
import sys
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

old = []
def load_images(image_dir, data):
    labels = []
    images = []
    gt_stats = []
    dirs = [image_dir + '/CS', image_dir + '/Healthy']
    for idx, image_dir in enumerate(dirs):
        for filename in sorted(os.listdir(image_dir)):
            if filename.endswith('.png'):
                print(filename)
                # Decode filename to extract sequence number, gender, and age
                # print(filename)
                seq_number = int(filename[:4])  # First 4 digits: sequence number

                # Find the corresponding row in the dataset
                row = data[data['Number'] == seq_number]

                if row.empty:
                    print(f"No matching row found for {filename}")
                    continue

             
                label_row = row.drop(columns=['Disease classification: 1. Cervical spondylosis; 2. Healthy', 'Number']).iloc[0]
                
                # Drop the Disease Classification column to use all other columns as label
                label_row = row.drop(columns=['Disease classification: 1. Cervical spondylosis; 2. Healthy', 'Number', 'Gender:female-1,male-0', 'Age', 'linear scale', 'Pixel equivalent', 'pixel distance', 'Curvature: 1.Lordotic, 2.Straight, 3.Sigmoid1, 4.Sigmoid2, 5.Kyphotic']).iloc[0]
                # 
                gt_stats.append(label_row.values)

                img_path = os.path.join(image_dir, filename)
                image = cv2.imread(img_path)
                image = cv2.resize(image, (224, 224))

                labels.append(idx)
                images.append(image)

    return np.array(gt_stats), np.array(labels), np.array(images)/1.0  # Normalize images

# File paths to the dataset and image directories
file_path = '/Users/srivatsavkannan/Datasets/C-Spine Xray/datasets.xlsx'
data = pd.read_excel(file_path, header=1).dropna()

train_image_dir = '/Users/srivatsavkannan/Datasets/FinalCervicalDataset/Train_Org'
val_image_dir = '/Users/srivatsavkannan/Datasets/FinalCervicalDataset/Val_Org'

# Load training and validation images and labels
X_train, y_train, images_train = load_images(train_image_dir, data)
X_val, y_val, images_val = load_images(val_image_dir, data)

smote = SMOTE(random_state=42)

# Apply ROS to the training and testing sets
X_train, y_train = smote.fit_resample(X_train, y_train)
X_val, y_val = smote.fit_resample(X_val, y_val)

print(X_train.shape)
print(X_val.shape)
print(images_train.shape)

print(y_train.shape)
print(y_val.shape)
print(images_val.shape)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

def build_saint_model(input_dim):
    inputs = layers.Input(shape=(input_dim,))

    # Feature Tokenization (Dense Layer for embedding features)
    x = layers.Dense(128, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    # Self-Attention Block
    for _ in range(2):  # 2 self-attention layers
        residual = x
        x = layers.Dense(128, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.3)(x)
        x = layers.Add()([x, residual])  # Skip connection

    # Output Layer
    outputs = layers.Dense(1, activation='sigmoid')(x)  # Binary classification

    model = models.Model(inputs=inputs, outputs=outputs)
    return model

def build_saint_with_cnn_model(tabular_input_dim, image_input_shape):
    # Tabular data input branch
    tabular_input = layers.Input(shape=(tabular_input_dim,), name="tabular_input")
    x = layers.Dense(128, activation='relu')(tabular_input)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    # Self-Attention Block for tabular data
    for _ in range(2):  # 2 self-attention layers
        residual = x
        x = layers.Dense(128, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.3)(x)
        x = layers.Add()([x, residual])  # Skip connection

    # Image data input branch
    image_input = layers.Input(shape=image_input_shape, name="image_input")
    y = layers.Conv2D(32, (3, 3), activation='relu')(image_input)
    y = layers.MaxPooling2D((2, 2))(y)
    y = layers.Conv2D(64, (3, 3), activation='relu')(y)
    y = layers.MaxPooling2D((2, 2))(y)
    y = layers.Flatten()(y)
    y = layers.Dense(128, activation='relu')(y)
    y = layers.Dropout(0.3)(y)

    # Combine tabular and image features (Late Fusion)
    combined = layers.Concatenate()([x, y])
    combined = layers.Dense(128, activation='relu')(combined)
    combined = layers.Dropout(0.3)(combined)

    # Output layer (Binary classification)
    outputs = layers.Dense(1, activation='sigmoid', name="output")(combined)

    # Create the model
    model = models.Model(inputs=[tabular_input, image_input], outputs=outputs)
    return model

def build_saint_with_efficientnet_model(tabular_input_dim, image_input_shape, embed_dim=128, num_heads=4, num_layers=2):
    
    # Tabular Data Processing with SAINT Architecture
    tabular_input = layers.Input(shape=(tabular_input_dim,), name="tabular_input")
    
    # Embedding Layer for tabular data (Represents numerical features as dense embeddings)
    tabular_embedding = layers.Dense(embed_dim, activation='relu')(tabular_input)
    
    # Transformer Blocks (Self-Attention + Intersample Attention)
    x = tabular_embedding
    for _ in range(num_layers):
        # Multi-Head Self-Attention Block
        attn_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)(x, x)
        attn_output = layers.Add()([x, attn_output])  # Skip connection
        attn_output = layers.LayerNormalization()(attn_output)
        
        # Feedforward & Normalization
        ff_output = layers.Dense(embed_dim, activation='relu')(attn_output)
        ff_output = layers.Dense(embed_dim)(ff_output)
        x = layers.Add()([attn_output, ff_output])  # Skip connection
        x = layers.LayerNormalization()(x)
        
        # Intersample Attention Block (attention across batch samples)
        intersample_attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)(x, x) # Intersample attention layer needed here, use the pytorch style pseudocode for proper attention heads
        x = layers.Add()([x, intersample_attn])
        x = layers.LayerNormalization()(x)

    # Image Processing with EfficientNetB7
    image_input = layers.Input(shape=image_input_shape, name="image_input")
    base_model = tf.keras.applications.EfficientNetB7(
        include_top=False,
        weights='imagenet',
        input_shape=image_input_shape
    )   
    base_model.trainable = False  # Freeze the base model

    y = base_model(image_input, training=False)
    y = layers.GlobalAveragePooling2D()(y)
    y = layers.Dense(512, activation='relu')(y)

    # Fusion of Tabular & Image Data
    combined = layers.Concatenate()([x, y])
    combined = layers.Dense(128, activation='relu')(combined)
    combined = layers.Dropout(0.3)(combined)

    # Output Layer (Binary Classification)
    outputs = layers.Dense(1, activation='sigmoid', name="output")(combined)

    # Create and return the model
    model = models.Model(inputs=[tabular_input, image_input], outputs=outputs)
    return model


In [ ]:
input_dim = X_train.shape[1]
image_input_dim = (224,224,3)
model = build_saint_model(input_dim)

model.compile(optimizer=tf.keras.optimizers.Adam(),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Train the model with class weights
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=100, restore_best_weights=True)
history = model.fit(X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=20,
                    batch_size=32,
                    callbacks=[early_stop],
                    verbose=1)

model.save("saint_118_smote.keras")

# Evaluate the model
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
test_loss, test_acc = model.evaluate(X_val, y_val, verbose=0)

In [ ]:
print(f"Training Accuracy: {train_acc}")
print(f"Testing Accuracy: {test_acc}")

# model = tf.keras.models.load_model("cspine118.keras")
# Predictions and metrics
y_pred_train = (model.predict(X_train) > 0.5).astype(int)
y_pred_test = (model.predict(X_val) > 0.5).astype(int)
y_pred_test_proba = model.predict(X_val).flatten()

# Compute metrics
print("Training Classification Report:\n", classification_report(y_train, y_pred_train, digits=4))
print("Testing Classification Report:\n", classification_report(y_val, y_pred_test, digits=4))

# Save the model


In [ ]:
print("Training Classification Report:\n", classification_report(y_train, y_pred_train, digits=4))
print("Testing Classification Report:\n", classification_report(y_val, y_pred_test, digits=4))